# Семинар 8 [17-25-2]. Случайные величины: функция распределения и плотность

В этом ноутбуке:

1. дискретная случайная величина: ряд распределения, многоугольник и «лесенка» $F(x)$;
2. проверка задач S08-01 – S08-03 в коде;
3. непрерывная случайная величина: плотность $f(x)$, функция $F(x)$ и вероятность как площадь;
4. проверка задач S08-05 и S08-06, показательный и равномерный законы;
5. гид по `scipy.stats`: какой метод что считает;
6. из данных в распределение: эмпирическая функция распределения и гистограмма;
7. оцениваем параметр распределения по данным;
8. как компьютер генерирует случайную величину с нужной $F(x)$;
9. практика: засуха, обеспеченность осадков и кривая $1-F(x)$.

Запускайте ячейки по порядку: **Среда выполнения → Выполнить все** или `Shift+Enter` по одной.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate

rng = np.random.default_rng(2026)   # фиксируем зерно, чтобы у всех получались одинаковые картинки
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Дискретная случайная величина

**Ряд распределения** — таблица «значение → вероятность». Сумма вероятностей равна 1.

**Функция распределения** на лекции определена как $F(x)=P(X<x)$: вероятность того, что величина *строго меньше* $x$. Для дискретной величины это сумма вероятностей всех значений левее $x$, и график — **лесенка**: ступенька высотой $p_i$ в каждой точке $x_i$.

> **Важно про `scipy`.** В `scipy` (и почти во всей англоязычной литературе) функция распределения определена как $P(X\le x)$. Для непрерывной величины разницы нет, потому что $P(X=x)=0$. Для дискретной разница есть ровно в точках $x_i$: по лекции $F(x_i)$ — значение *до* ступеньки, а `cdf(x_i)` в `scipy` — значение *после* неё.

Возьмём ряд из задачи S08-03 (задача 4.1 методички), $p_3=0{,}4$ находим из нормировки.

In [ ]:
x_vals = np.array([-2, 0, 2, 5])
p_vals = np.array([0.3, 0.2, 0.0, 0.1])
p_vals[2] = 1 - p_vals.sum()            # p3 из условия нормировки
print("p3 =", round(p_vals[2], 4), "| сумма вероятностей:", p_vals.sum())

# Свой дискретный закон в scipy: rv_discrete(values=(значения, вероятности))
X = stats.rv_discrete(values=(x_vals, p_vals))

def F_lecture(dist, x, eps=1e-9):
    # F(x) = P(X < x): значение scipy-cdf чуть левее точки x
    return dist.cdf(np.asarray(x) - eps)

table = pd.DataFrame({
    "x": x_vals,
    "P(X = x)": X.pmf(x_vals),
    "F(x) = P(X < x), как на лекции": F_lecture(X, x_vals).round(4),
    "scipy cdf(x) = P(X <= x)": X.cdf(x_vals),
}).set_index("x")
display(table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Многоугольник распределения
axes[0].vlines(x_vals, 0, p_vals, lw=6, alpha=0.4)
axes[0].plot(x_vals, p_vals, "o-", color="black")
axes[0].set_title("Многоугольник распределения")
axes[0].set_xlabel("x"); axes[0].set_ylabel("P(X = x)")

# Лесенка F(x) = P(X < x): на каждом полуинтервале (x_i; x_{i+1}] значение постоянно
edges = np.concatenate([[x_vals[0] - 1.5], x_vals, [x_vals[-1] + 1.5]])
levels = np.concatenate([[0], np.cumsum(p_vals)])
for a, b, Fv in zip(edges[:-1], edges[1:], levels):
    axes[1].hlines(Fv, a, b, lw=2.5)
axes[1].plot(x_vals, levels[:-1], "o", color="C0")                  # F(x_i) — нижнее значение (закрашенная точка)
axes[1].plot(x_vals, levels[1:], "o", mfc="white", color="C0")      # верхнее значение не достигается в x_i
axes[1].set_title("Функция распределения F(x) = P(X < x)")
axes[1].set_xlabel("x"); axes[1].set_ylabel("F(x)")
plt.tight_layout(); plt.show()

**Как читать лесенку.** Высота ступеньки в точке $x_i$ равна $P(X=x_i)$. Между значениями $F$ не меняется: вероятность «живёт» только в точках. Вероятность попасть в полуинтервал: $P(a\le X<b)=F(b)-F(a)$.

In [ ]:
# P(0 <= X < 5) = F(5) - F(0)
print("P(0 <= X < 5) =", round(F_lecture(X, 5) - F_lecture(X, 0), 4))
print("Проверка суммой:", p_vals[1] + p_vals[2])

## 2. Проверяем задачи S08-01 – S08-03

**S08-01 (4.6).** Три стрелка, вероятности попадания $0{,}5$, $0{,}8$, $0{,}7$. Ряд можно получить перебором всех $2^3=8$ исходов: так компьютер повторяет то, что мы делали у доски.

In [ ]:
from itertools import product

p_hit = [0.5, 0.8, 0.7]
law = {}
for outcome in product([0, 1], repeat=3):          # 1 — попадание
    prob = np.prod([p if o else 1 - p for o, p in zip(outcome, p_hit)])
    law[sum(outcome)] = law.get(sum(outcome), 0) + prob

X1 = stats.rv_discrete(values=(list(law), list(law.values())))
print("Ряд распределения:", {k: round(v, 4) for k, v in sorted(law.items())})
print("P(X >= 2) =", round(X1.sf(1), 4), " (sf(k) = P(X > k))")

# Симуляция: 100 000 серий по три выстрела
volleys = (rng.random((100_000, 3)) < p_hit).sum(axis=1)
print("Частоты в симуляции:", np.round(np.bincount(volleys, minlength=4) / len(volleys), 4))

**S08-02 (4.3).** В коробке 9 фломастеров, 2 не пишут; берут 3. Число пишущих среди взятых имеет **гипергеометрическое** распределение: `stats.hypergeom(M=всего, n=пишущих, N=берём)`.

In [ ]:
X2 = stats.hypergeom(M=9, n=7, N=3)
k = np.arange(1, 4)                   # хотя бы один пишущий будет всегда: не пишущих только 2
print(pd.DataFrame({"k": k, "P(X = k)": X2.pmf(k).round(4), "в долях 1/84": (X2.pmf(k) * 84).round(1)}).set_index("k"))
print("P(X >= 2) =", round(X2.sf(1), 4))

**S08-03 (4.1)** уже разобрана в разделе 1.

## 3. Непрерывная случайная величина

У непрерывной величины $P(X=a)=0$ для любого $a$, поэтому ряд не работает. Вместо него — **плотность** $f(x)=F'(x)$:

$$
P(a<X<b)=\int_a^b f(x)\,dx = F(b)-F(a),\qquad F(x)=\int_{-\infty}^x f(t)\,dt,\qquad \int_{-\infty}^{+\infty} f(x)\,dx=1.
$$

Вероятность — это **площадь под кривой плотности**, а значение $F(x)$ — площадь левее $x$.

Задача S08-04 (5.3): $f(x)=c\,x$ на $(2;4]$ и $0$ вне.

In [ ]:
area, _ = integrate.quad(lambda x: x, 2, 4)
c = 1 / area
print(f"Интеграл x от 2 до 4 = {area:.4f}  ->  c = {c:.5f} = 1/6")

def f(x):
    x = np.asarray(x, dtype=float)
    return np.where((x > 2) & (x <= 4), c * x, 0.0)

def F(x):
    x = np.clip(np.asarray(x, dtype=float), 2, 4)
    return (x**2 - 4) / 12

print("P(X > 3) через интеграл:", round(integrate.quad(f, 3, 4)[0], 5))
print("P(X > 3) = 1 - F(3):    ", round(1 - F(3), 5), "= 7/12 =", round(7 / 12, 5))

In [ ]:
xs = np.linspace(1.5, 4.5, 500)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(xs, f(xs), color="C0", lw=2)
mask = (xs > 3) & (xs < 4)
axes[0].fill_between(xs[mask], f(xs[mask]), alpha=0.35, label=f"P(X > 3) = площадь = {1 - F(3):.4f}")
axes[0].set_title("Плотность f(x)"); axes[0].set_xlabel("x"); axes[0].legend()

axes[1].plot(xs, F(xs), color="C1", lw=2)
axes[1].plot([3, 3], [0, F(3)], "k:"); axes[1].plot([1.5, 3], [F(3), F(3)], "k:")
axes[1].annotate("", xy=(3.15, 1), xytext=(3.15, F(3)), arrowprops=dict(arrowstyle="<->", color="C0"))
axes[1].text(3.25, (1 + F(3)) / 2, "1 - F(3)", color="C0")
axes[1].set_title("Функция распределения F(x)"); axes[1].set_xlabel("x")
plt.tight_layout(); plt.show()

**Связь двух графиков.** Площадь закрашенной полосы слева равна длине стрелки справа. Там, где плотность выше, $F(x)$ растёт круче: $f$ — это скорость роста $F$.

### Плотность — не вероятность

Плотность может быть больше единицы. Для равномерного закона на $[0;\,0{,}1]$ плотность равна $10$, а вероятности всё равно не больше 1, потому что ширина отрезка мала.

In [ ]:
U = stats.uniform(loc=0, scale=0.1)        # равномерный на [loc; loc + scale]
print("f(0,05) =", U.pdf(0.05), "   P(0 < X < 0,1) =", U.cdf(0.1) - U.cdf(0))

# А вероятность попасть в одну точку равна нулю: сужаем интервал вокруг 0,05
for h in [0.01, 0.001, 0.0001]:
    print(f"P(0,05 - {h} < X < 0,05 + {h}) = {U.cdf(0.05 + h) - U.cdf(0.05 - h):.5f}")

## 4. Проверяем задачи S08-05 и S08-06

**S08-05 (5.1).** $F(x)=\dfrac{2x}{x+1}$ на $(0;1]$. Плотность — производная; `sympy` возьмёт её символьно.

In [ ]:
import sympy as sp

x = sp.symbols("x")
F5 = 2 * x / (x + 1)
f5 = sp.simplify(sp.diff(F5, x))
print("f(x) =", f5, "на (0; 1]")
print("Проверка нормировки: интеграл f от 0 до 1 =", sp.integrate(f5, (x, 0, 1)))
# На (-1/2; 0] плотность равна 0, поэтому P(-1/2 < X < 1/2) = F(1/2) - F(0)
print("P(-1/2 < X < 1/2) =", F5.subs(x, sp.Rational(1, 2)) - F5.subs(x, 0))

**S08-06 (5.9).** $F(x)=\frac1{16}(x^2-6x+9)=\frac{(x-3)^2}{16}$ на $(3;7)$.

In [ ]:
F6 = (x - 3)**2 / 16
print("f(x) =", sp.factor(sp.diff(F6, x)), "на (3; 7)")
print("P(X < 5) =", F6.subs(x, 5))
print("P(X >= 7) = 1 - F(7) =", 1 - F6.subs(x, 7))
print("f(7) =", sp.diff(F6, x).subs(x, 7))

Два стандартных закона, которые встретятся в ноутбуке дальше.

**Показательный** закон: $F(t)=1-e^{-\lambda t}$ при $t\ge0$, например время безотказной работы прибора.

> В `scipy` показательный закон задаётся не через $\lambda$, а через `scale` $=1/\lambda$: `stats.expon(scale=2)` при $\lambda=0{,}5$. Это самая частая ошибка при переходе от учебника к коду.

**Равномерный** закон на $[a;b]$: `stats.uniform(loc=a, scale=b-a)`.

In [ ]:
T = stats.expon(scale=1 / 0.5)
print("Показательный, λ = 0,5: f(1) =", round(T.pdf(1), 4), "  P(T > 2) =", round(T.sf(2), 4), "= e^{-1}")
W = stats.uniform(loc=0, scale=30)
print("Равномерный на [0; 30]: f(x) =", round(W.pdf(10), 4), "  P(5 < X < 20) =", round(W.cdf(20) - W.cdf(5), 4))

## 5. Гид по `scipy.stats`

Любое распределение в `scipy` — объект с одинаковым набором методов. Сначала «замораживаем» параметры, потом спрашиваем.

| Метод | Что считает | Формула |
|---|---|---|
| `pmf(k)` | вероятность значения (только дискретные) | $P(X=k)$ |
| `pdf(x)` | плотность (только непрерывные) | $f(x)$ |
| `cdf(x)` | функция распределения в соглашении `scipy` | $P(X\le x)$ |
| `sf(x)` | «хвост», функция выживания | $1-$`cdf(x)` $=P(X>x)$ |
| `ppf(q)` | квантиль: обратная к `cdf` | такое $x$, что `cdf(x)` $=q$ |
| `isf(q)` | обратная к `sf` | такое $x$, что $P(X>x)=q$ |
| `rvs(size, random_state)` | случайная выборка | симуляция |

| Закон | Учебник | `scipy` |
|---|---|---|
| биномиальный | $n$, $p$ | `stats.binom(n, p)` |
| Пуассона | $\lambda$ | `stats.poisson(mu=λ)` |
| гипергеометрический | всего $N$, «особых» $M$, берём $n$ | `stats.hypergeom(M=N, n=M, N=n)` (буквы перепутаны!) |
| свой ряд | таблица | `stats.rv_discrete(values=(x, p))` |
| равномерный на $[a;b]$ | $a$, $b$ | `stats.uniform(loc=a, scale=b-a)` |
| показательный | $\lambda$ | `stats.expon(scale=1/λ)` |

Ниже — одна и та же величина глазами всех методов.

In [ ]:
T = stats.expon(scale=2)            # показательный закон, λ = 0,5
print("pdf(1) =", T.pdf(1))
print("cdf(1) =", T.cdf(1), " sf(1) =", T.sf(1), " сумма =", T.cdf(1) + T.sf(1))
print("ppf(cdf(1)) =", T.ppf(T.cdf(1)), " — квантиль возвращает нас в точку 1")
print("Отрезок, куда T попадает с вероятностью 0,9 (по 5 % отрезаем с краёв):", np.round(T.ppf([0.05, 0.95]), 3))
print("Пять случайных значений:", np.round(T.rvs(size=5, random_state=rng), 2))

**Свой непрерывный закон.** Если нужного закона в `scipy` нет, его можно описать, задав только плотность (или только $F$). Остальные методы `scipy` досчитает сам, численно. Возьмём плотность из S08-04.

In [ ]:
class Task504(stats.rv_continuous):
    # Плотность из S08-04: f(x) = x/6 на (2; 4]
    def _pdf(self, x):
        return x / 6

G = Task504(a=2, b=4, name="task_5_3")
print("P(X > 3) =", round(G.sf(3), 5))
print("Точка, левее которой половина вероятности (медиана):", round(G.ppf(0.5), 4))

**Вопрос:** проверьте медиану руками: решите уравнение $F(x)=\frac{x^2-4}{12}=0{,}5$. Почему она правее середины отрезка $x=3$?

## 6. Из данных в распределение

На практике формулу $F(x)$ нам никто не даёт: у нас есть только наблюдения. Как по ним восстановить распределение?

- **Эмпирическая функция распределения** $F_n(x)$ — доля наблюдений, меньших $x$. Это та же лесенка, что в разделе 1, только каждая ступенька высотой $1/n$.
- **Гистограмма плотности** — столбики высотой «доля наблюдений в столбике / ширина столбика», так что общая площадь равна 1. Это приближение к $f(x)$.

Сгенерируем $n$ значений величины из S08-04 и посмотрим, как по ним восстанавливается распределение.

In [ ]:
def ecdf(sample, x):
    # Эмпирическая функция распределения: доля наблюдений, строго меньших x
    sample = np.sort(sample)
    return np.searchsorted(sample, x, side="left") / len(sample)

xs = np.linspace(1.8, 4.2, 400)
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for j, n in enumerate([20, 200, 5000]):
    sample = G.rvs(size=n, random_state=rng)
    axes[0, j].hist(sample, bins=16, range=(2, 4), density=True, alpha=0.5, label="гистограмма")
    axes[0, j].plot(xs, G.pdf(xs), "k", lw=2, label="плотность f(x)")
    axes[0, j].set_title(f"n = {n}")
    axes[1, j].step(xs, ecdf(sample, xs), where="post", label="эмпирическая F_n(x)")
    axes[1, j].plot(xs, G.cdf(xs), "k--", lw=1.5, label="F(x)")
    gap = np.abs(ecdf(sample, xs) - G.cdf(xs)).max()
    axes[1, j].set_title(f"макс. расхождение = {gap:.3f}")
axes[0, 0].legend(); axes[1, 0].legend()
plt.tight_layout(); plt.show()

**Что видно.** С ростом $n$ эмпирическая функция прилипает к настоящей $F(x)$: максимальное расхождение убывает примерно как $1/\sqrt{n}$ (теорема Гливенко — Кантелли). Гистограмма при малом $n$ «скачет» сильнее, чем лесенка: оценить $F$ по данным проще, чем $f$.

### Ширина столбика решает

Одни и те же 300 наблюдений с разным числом столбиков:

In [ ]:
sample = G.rvs(size=300, random_state=rng)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, bins in zip(axes, [3, 15, 120]):
    ax.hist(sample, bins=bins, range=(2, 4), density=True, alpha=0.5)
    ax.plot(xs, G.pdf(xs), "k", lw=2)
    ax.set_title(f"{bins} столбиков")
plt.tight_layout(); plt.show()

**Вопрос:** что «врёт» на левой картинке, а что на правой? Почему у эмпирической функции распределения такой проблемы нет?

## 7. Оцениваем параметр по данным

Пусть время безотказной работы приборов показательный, но $\lambda$ неизвестна. Мы наблюдали 50 приборов. Как оценить $\lambda$, **используя только функцию распределения**?

Идея: $P(T>t)=e^{-\lambda t}$. Долю приборов, проработавших дольше $t$, мы видим в данных, значит

$$
\hat\lambda=-\frac{\ln\big(\text{доля приборов с } T>t\big)}{t}.
$$

In [ ]:
LAMBDA_TRUE = 0.5
pumps = stats.expon(scale=1 / LAMBDA_TRUE).rvs(size=50, random_state=rng)

t0 = 2.0
share_alive = np.mean(pumps > t0)
lam_hat = -math.log(share_alive) / t0
print(f"Проработали дольше {t0} лет: {share_alive:.2f}  ->  оценка λ = {lam_hat:.3f} (истинное {LAMBDA_TRUE})")

xs_t = np.linspace(0, 10, 300)
plt.step(xs_t, ecdf(pumps, xs_t), where="post", label="данные: 50 приборов")
plt.plot(xs_t, 1 - np.exp(-lam_hat * xs_t), label=f"F(t) с оценкой λ = {lam_hat:.2f}")
plt.plot(xs_t, 1 - np.exp(-LAMBDA_TRUE * xs_t), "k--", label="истинная F(t)")
plt.axvline(t0, color="gray", ls=":")
plt.xlabel("годы"); plt.legend(); plt.show()

**Насколько точна оценка?** Повторим «наблюдение за 50 приборами» 2000 раз и посмотрим, как разбросаны оценки. Затем то же для 500 приборов.

In [ ]:
def estimate_lambda(n, t0=2.0, reps=2000):
    data = stats.expon(scale=1 / LAMBDA_TRUE).rvs(size=(reps, n), random_state=rng)
    share = np.mean(data > t0, axis=1)
    share = np.clip(share, 1 / n, 1 - 1 / n)          # защита от ln(0)
    return -np.log(share) / t0

fig, ax = plt.subplots()
for n in [50, 500]:
    est = estimate_lambda(n)
    lo, hi = np.quantile(est, [0.05, 0.95])
    ax.hist(est, bins=50, density=True, alpha=0.5, label=f"n = {n}: 90 % оценок в [{lo:.2f}; {hi:.2f}]")
ax.axvline(LAMBDA_TRUE, color="black", lw=2, label="истинное λ")
ax.set_xlabel("оценка λ"); ax.legend(); plt.show()

**Вывод.** Оценка сама по себе случайная величина со своим распределением. При $n=50$ она промахивается заметно, при $n=500$ отрезок, куда попадают 90 % оценок, сужается примерно в $\sqrt{10}\approx3$ раза.

**Ещё два способа оценить параметр по $F$:**

- *Равномерный закон на $[0;b]$:* $b$ неизвестно. Самое большое наблюдение всегда немного меньше $b$, поэтому берут оценку $\max\cdot\frac{n+1}{n}$.
- *Квантиль:* медиана закона (точка, где $F(x)=0{,}5$) оценивается медианой выборки, то есть точкой, где эмпирическая функция переходит через $0{,}5$.

In [ ]:
# Равномерный закон на [0; b], b неизвестно (истинное значение 30)
waits = stats.uniform(0, 30).rvs(size=20, random_state=rng)
print("max =", round(waits.max(), 2), " -> оценка b =", round(waits.max() * 21 / 20, 2))

# Медиана по выборке из 40 значений против истинной медианы закона S08-04
smp = G.rvs(size=40, random_state=rng)
print("Медиана по выборке:", round(np.median(smp), 3), " истинная:", round(G.ppf(0.5), 3))

## 8. Как компьютер получает величину с нужной $F(x)$

Генератор умеет только одно: выдавать $U$, равномерное на $[0;1]$. Чтобы получить величину с функцией распределения $F$, достаточно взять

$$
X=F^{-1}(U).
$$

Почему: $P(X<x)=P(F^{-1}(U)<x)=P(U<F(x))=F(x)$. Для показательного закона $F^{-1}(u)=-\ln(1-u)/\lambda$.

In [ ]:
u = rng.random(20_000)
lam = 0.5
x_inv = -np.log(1 - u) / lam            # обратная функция к F(t) = 1 - e^{-λt}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(u, bins=40, density=True, alpha=0.5); axes[0].set_title("U: равномерные на [0; 1]")
axes[1].hist(x_inv, bins=60, range=(0, 12), density=True, alpha=0.5, label="F⁻¹(U)")
tt = np.linspace(0, 12, 300)
axes[1].plot(tt, stats.expon(scale=1 / lam).pdf(tt), "k", lw=2, label="плотность показательного закона")
axes[1].set_title("X = F⁻¹(U)"); axes[1].legend()
plt.tight_layout(); plt.show()

**Вопрос:** для дискретной величины из раздела 1 (задача S08-03) как превратить $U$ в значение $X$? Подсказка: разбейте отрезок $[0;1]$ на кусочки длиной $p_i$.

In [ ]:
# Ответ на вопрос: кусочки [0; 0,3), [0,3; 0,5), [0,5; 0,9), [0,9; 1) -> номер кусочка -> значение x_i
cut = np.cumsum(p_vals)
idx = np.searchsorted(cut, rng.random(100_000), side="right")
sim_x = x_vals[idx]
print("Частоты:", {int(v): round(np.mean(sim_x == v), 3) for v in x_vals}, " теория:", p_vals)

## 9. Практика: засуха и обеспеченность осадков

В агрометеорологии и гидрологии говорят об **обеспеченности** осадков: вероятности того, что за сезон выпадет *не меньше* данного количества. Это $1-F(x)$. «Осадки 75 %-й обеспеченности» — уровень, который превышается в 3 годах из 4, по нему планируют орошение. Вероятность засухи — $F(\text{порог})$.

Ниже 40 «лет наблюдений» за осадками вегетационного периода (мм). Данные сгенерированы, но похожи на реальные для средней полосы.

In [ ]:
rain = np.round(stats.gamma(a=9, scale=32).rvs(size=40, random_state=np.random.default_rng(7)))
print("Осадки по годам, мм:", rain.astype(int))

DROUGHT = 200
print(f"Доля лет с осадками меньше {DROUGHT} мм (оценка F({DROUGHT})): {np.mean(rain < DROUGHT):.3f}")
print("Осадки 75 %-й обеспеченности (превышаются в 75 % лет):", np.quantile(rain, 0.25), "мм")

xr = np.linspace(rain.min() - 20, rain.max() + 20, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].step(xr, ecdf(rain, xr), where="post")
axes[0].axvline(DROUGHT, color="red", ls="--", label="порог засухи")
axes[0].set_title("Эмпирическая F(x)"); axes[0].set_xlabel("мм"); axes[0].legend()
axes[1].step(xr, 1 - ecdf(rain, xr), where="post", color="C2")
axes[1].axhline(0.75, color="gray", ls=":")
axes[1].set_title("Кривая обеспеченности 1 − F(x)"); axes[1].set_xlabel("мм")
plt.tight_layout(); plt.show()

**Вопросы:**

1. В наших 40 годах нет ни одного года с осадками меньше 100 мм. Значит ли это, что вероятность такой засухи равна нулю?
2. Как изменилась бы кривая обеспеченности, если бы климат стал суше и все значения сдвинулись влево на 30 мм?

---

## Итоги

1. Дискретная величина задаётся рядом распределения, непрерывная — плотностью. Функция распределения $F(x)$ есть у обеих: у дискретной это лесенка, у непрерывной — гладкая кривая.
2. Вероятность попасть в интервал равна $F(b)-F(a)$; для непрерывной величины это ещё и площадь под плотностью. Плотность — не вероятность и может быть больше 1.
3. По данным $F(x)$ восстанавливается эмпирической функцией распределения, $f(x)$ — гистограммой; параметры закона можно оценить, приравняв теоретическую $F$ к наблюдаемой доле.
4. В `scipy`: `pmf`/`pdf`, `cdf` (это $P(X\le x)$!), `sf`, `ppf`, `rvs`; у показательного закона `scale = 1/λ`.